In [0]:
%sql
DROP VOLUME IF EXISTS northstar.bronze.landing;

CREATE EXTERNAL VOLUME northstar.bronze.landing
LOCATION 'abfss://landing@arnalla.dfs.core.windows.net/';

In [0]:
from pyspark.sql import functions as F

customers = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("header", "true")
    .option("cloudFiles.schemaLocation", "/Volumes/northstar/bronze/landing/_meta/customers_schema")
    .option("pathGlobFilter", "*customers*.csv")          # only the customers file
    .load("/Volumes/northstar/bronze/landing/")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path")))

(customers.writeStream
    .option("checkpointLocation", "/Volumes/northstar/bronze/landing/_meta/customers_checkpoint")
    .trigger(availableNow=True)                            # runs once, then stops (cost-safe)
    .toTable("northstar.bronze.customers_raw"))

In [0]:
%sql
SELECT COUNT(*) AS total_customers FROM northstar.bronze.customers_raw;


In [0]:
from pyspark.sql import functions as F

landing = "/Volumes/northstar/bronze/landing"

# Map: bronze table name  ->  source file pattern
tables = {
    "customers":   "*customers_dataset.csv",
    "geolocation": "*geolocation_dataset.csv",
    "order_items": "*order_items_dataset.csv",
    "payments":    "*order_payments_dataset.csv",
    "reviews":     "*order_reviews_dataset.csv",
    "orders":      "*orders_dataset.csv",
    "products":    "*products_dataset.csv",
    "sellers":     "*sellers_dataset.csv",
    "categories":  "*category_name_translation.csv",
}

for name, pattern in tables.items():
    q = (spark.readStream.format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("header", "true")
        .option("cloudFiles.schemaLocation", f"{landing}/_meta/{name}_schema")
        .option("pathGlobFilter", pattern)
        .load(landing)
        .withColumn("_ingested_at", F.current_timestamp())
        .withColumn("_source_file", F.col("_metadata.file_path"))
        .writeStream
        .option("checkpointLocation", f"{landing}/_meta/{name}_checkpoint")
        .trigger(availableNow=True)
        .toTable(f"northstar.bronze.{name}_raw"))
    q.awaitTermination()           # finish this table before the next
    print(f"✅ Loaded northstar.bronze.{name}_raw")

In [0]:
%sql
SHOW TABLES IN northstar.bronze;

In [0]:
%sql
SELECT 'customers' t, COUNT(*) n FROM northstar.bronze.customers_raw
UNION ALL SELECT 'orders',   COUNT(*) FROM northstar.bronze.orders_raw
UNION ALL SELECT 'reviews',  COUNT(*) FROM northstar.bronze.reviews_raw
UNION ALL SELECT 'order_items', COUNT(*) FROM northstar.bronze.order_items_raw
UNION ALL SELECT 'products', COUNT(*) FROM northstar.bronze.products_raw
ORDER BY t;

In [0]:
from pyspark.sql import functions as F

reviews_raw = (spark.read.format("csv")
    .option("header", "true")
    .option("multiLine", "true")     # ← review text spans multiple lines
    .option("quote", '"')            # ← fields wrapped in quotes
    .option("escape", '"')           # ← quotes inside text are doubled
    .load("/Volumes/northstar/bronze/landing/olist_order_reviews_dataset.csv")
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.col("_metadata.file_path")))

reviews_raw.write.mode("overwrite").saveAsTable("northstar.bronze.reviews_raw")
print(f"✅ re-ingested bronze.reviews_raw — {reviews_raw.count()} rows")